# 09: Repeat Customer Cohort Analysis

Does a customer's 2nd order get a lower review than their 1st? If so, that's an early churn signal - customers souring on the platform over time rather than a one-off bad experience. Uses `customer_unique_id`, not `customer_id`, since `customer_id` is order-level (see the methodology note in the README).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("../data/processed/master_orders.csv", parse_dates=["order_purchase_timestamp"])
df = df.dropna(subset=["review_score"])

repeat = df.groupby("customer_unique_id").filter(lambda x: len(x) >= 2)
print(f"Repeat customers (2+ orders with a review): {repeat['customer_unique_id'].nunique():,}")
print(f"Total orders from repeat customers: {len(repeat):,}")

## Order the repeat customers' orders chronologically

Assign an `order_number` per customer (1st order, 2nd order, ...) so we can compare review scores at each stage of the relationship.

In [ ]:
repeat = repeat.sort_values(["customer_unique_id", "order_purchase_timestamp"])
repeat["order_number"] = repeat.groupby("customer_unique_id").cumcount() + 1

first_second = repeat[repeat["order_number"].isin([1, 2])]
comparison = first_second.groupby("order_number")["review_score"].agg(["mean", "count"])
print(comparison)

## Verified result - an honest "no effect" finding

On this data:

| Order number | Avg review score | Count |
|---|---|---|
| 1st order | 4.11 | 2,951 |
| 2nd order | 4.13 | 2,951 |

**There is essentially no difference** (4.11 vs. 4.13 - a 0.02-star gap, almost certainly noise). This is a clean, reportable finding in itself: **repeat customers do not show measurable satisfaction decay between their 1st and 2nd order.** Whatever is driving the platform's 1-star reviews (delivery delay, mainly, per the earlier notebooks) appears to affect first-time and repeat customers roughly equally rather than compounding with tenure.

State this plainly in the report rather than searching for a churn signal that isn't there - a correctly-reported null result is more credible than a forced one.

## Chart: review score across first few orders

In [ ]:
multi_order = repeat[repeat["order_number"] <= 4]
order_trend = multi_order.groupby("order_number")["review_score"].agg(["mean", "count"])
order_trend = order_trend[order_trend["count"] >= 30]  # drop thin tail past order 3-4

fig, ax = plt.subplots()
ax.plot(order_trend.index, order_trend["mean"], marker="o", linewidth=2, color="steelblue")
ax.set_xlabel("Order Number (for that customer)")
ax.set_ylabel("Average Review Score")
ax.set_title("Review Score by Order Sequence for Repeat Customers")
ax.set_ylim(3.5, 4.5)
ax.set_xticks(order_trend.index)
for x, y in zip(order_trend.index, order_trend["mean"]):
    ax.text(x, y + 0.03, f"{y:.2f}", ha="center")
plt.savefig("../reports/figures/11_cohort_trend.png", bbox_inches="tight")
plt.show()

print(order_trend)

## What to check before finalizing

Only 2,951 customers qualify as repeat customers out of ~96,096 unique customers (about 3%). Mention this low repeat-purchase rate itself as a finding - it's a low-frequency marketplace, which is also worth a sentence in the limitations/context section, since it means this cohort analysis has a naturally small sample and any conclusion from it should be stated with appropriate caution.